In [1]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('data/claims_train.csv')

X = df[['Density', 'DrivAge', 'VehAge', 'Exposure', 'VehPower', 'BonusMalus', 'VehGas']]
X = pd.get_dummies(X, drop_first=True)
y = df['ClaimNb']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
clf = DecisionTreeRegressor(random_state=42, min_samples_leaf=5, max_depth=8)
clf.fit(x_train, y_train)
y_hat = clf.predict(x_test)

print('MSE:', mean_squared_error(y_test, y_hat))
print(f"R² Score: {r2_score(y_test, y_hat):.4f}")

print(y_hat[:100])

MSE: 0.056147462895605155
R² Score: 0.0271
[0.03637858 0.00849858 0.04939815 0.07003646 0.10549828 0.20992761
 0.02161124 0.07003646 0.02699037 0.02161124 0.17671233 0.02215146
 0.04939815 0.07003646 0.28810021 0.05945755 0.05945755 0.04503676
 0.03079409 0.04939815 0.10549828 0.21822542 0.02699037 0.04217404
 0.06412781 0.01358138 0.04652326 0.03079409 0.05803794 0.02699037
 0.05945755 0.06412781 0.02161124 0.12875536 0.06412781 0.04503676
 0.07104758 0.04285588 0.04939815 0.07003646 0.05945755 0.01358138
 0.03079409 0.03214696 0.02215146 0.04939815 0.01358138 0.04939815
 0.06412781 0.03560881 0.04939815 0.04939815 0.07003646 0.02215146
 0.00929978 0.06412781 0.05305117 0.01460849 0.06412781 0.06412781
 0.01358138 0.06412781 0.02215146 0.04217404 0.05803794 0.00929978
 0.07003646 0.04495504 0.06412781 0.09252669 0.00849858 0.01358138
 0.12907117 0.03560881 0.04939815 0.07003646 0.00929978 0.01629431
 0.00929978 0.04503676 0.02215146 0.04939815 0.04503676 0.00929978
 0.18508997 0.04217

In [27]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

def rss(y):
    return len(y) * np.var(y) if len(y) > 0 else 0

def proportions(region):
    """Return class 0 and 1 proportions in a region"""
    if len(region) == 0:
        return (0, 0)
    count0 = sum(1 for _, ci in region if ci == 0)
    count1 = sum(1 for _, ci in region if ci == 1)
    total = count0 + count1
    return (count0 / total, count1 / total)

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def best_split(X, y):
    n_samples, n_features = X.shape
    best_feature, best_threshold = None, None
    best_rss = float("inf")

    for feature in range(n_features):
        values = X[:, feature]
        thresholds = np.unique(values)
        for t in thresholds:
            left_idx = values < t
            right_idx = values >= t
            if np.sum(left_idx) == 0 or np.sum(right_idx) == 0:
                continue

            rss_left = rss(y[left_idx])
            rss_right = rss(y[right_idx])
            total_rss = rss_left + rss_right

            if total_rss < best_rss:
                best_rss = total_rss
                best_feature = feature
                best_threshold = t

    return best_feature, best_threshold

# --- Recursive Tree Builder ---
def build_tree(X, y, depth=0, max_depth=8, min_leaves = 5):
    
    if depth >= max_depth or len(np.unique(y)) == 1 or len(y) < min_leaves:
        return Node(value=np.mean(y))

    feature, threshold = best_split(X, y)
    if feature is None:
        return Node(value=np.mean(y))

    left_idx = X[:, feature] < threshold
    right_idx = X[:, feature] >= threshold

    left = build_tree(X[left_idx], y[left_idx], depth+1, max_depth)
    right = build_tree(X[right_idx], y[right_idx], depth+1, max_depth)
    return Node(feature, threshold, left, right)

# --- Prediction ---
def predict_one(node, x):
    if node.value is not None:
        return node.value
    if x[node.feature] < node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict(node, X):
    return np.array([predict_one(node, x) for x in X])

df = pd.read_csv('data/claims_train.csv')

X = df[['Density', 'DrivAge', 'VehAge', 'Exposure', 'VehPower', 'BonusMalus', 'VehGas']]
X = pd.get_dummies(X, drop_first=True, dtype="int")
y = df['ClaimNb']

X = X.to_numpy()
y = y.to_numpy()

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree = build_tree(x_train, y_train)
y_pred = predict(tree, x_test)

print('MSE:', mean_squared_error(y_test, y_pred))
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")

MSE: 0.056324546777877425
R² Score: 0.0241


In [ ]:
""" 
2mins and 40 seconds
True/False encoding
MSE: 0.056324546777877425
R² Score: 0.0241


Int encoding:
MSE: 0.056324546777877425
R² Score: 0.0241

"""